In [2]:
# Importing the pandas library
import pandas as pd

# Reading the CSV file
# Replace 'your_file.csv' with the path to your CSV file
csv_file_path = 'car_data.csv'
df = pd.read_csv(csv_file_path)

# Display the first few rows of the data
print(df.head())
print(df['Price'])


  Version  Power (hp)  Consumption (l/100 km) Interior  \
0     NaN         NaN                     NaN      NaN   
1     NaN         NaN                     NaN      NaN   
2     GLI         NaN                     NaN      NaN   
3     GLI         NaN                     NaN      NaN   
4    2021         NaN                     NaN      NaN   

                          Location Registration city Number of doors Assembly  \
0  Bahria Town Phase 8, Rawalpindi            Lahore             NaN    Local   
1  Bahria Town Phase 8, Rawalpindi            Lahore             NaN    Local   
2  Bahria Town Phase 8, Rawalpindi         Islamabad             NaN    Local   
3  Bahria Town Phase 8, Rawalpindi         Islamabad             NaN    Local   
4  Bahria Town Phase 8, Rawalpindi         Islamabad             NaN    Local   

  Source                      Model Body Type Car documents  \
0    NaN  Civic VTi Oriel Prosmatec     Sedan      Original   
1    NaN  Civic VTi Oriel Prosmatec   

In [3]:
# Remove 'Rs' and commas, convert to float, and replace invalid entries with NaN
df['Price'] = df['Price'].replace({'Rs': '', ',': ''}, regex=True)

# Convert to numeric and replace invalid values (strings) with NaN
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Replace NaN values with 0
df['Price'].fillna(0, inplace=True)

# Convert the 'Price' column to float
df['Price'] = df['Price'].astype(float)

# Print the updated 'Price' column and its datatype
print("\nUpdated 'Price' column:")
print(df['Price'])

print("\nDatatype of 'Price' column after conversion:", df['Price'].dtype)



Updated 'Price' column:
0        3850000.0
1        3850000.0
2        3985000.0
3        3985000.0
4        3980000.0
           ...    
12553    4450000.0
12554    3150000.0
12555    3150000.0
12556    2250000.0
12557    2250000.0
Name: Price, Length: 12558, dtype: float64

Datatype of 'Price' column after conversion: float64


C:\Users\warda\AppData\Local\Temp\ipykernel_20116\350000793.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Price'].fillna(0, inplace=True)


In [4]:
# Print the count of rows before removing duplicates
print("Count of rows before removing duplicates:", len(df))

# Remove rows with duplicate values across all columns
df = df.drop_duplicates()

# Print the count of rows after removing duplicates
print("Count of rows after removing duplicates:", len(df))

Count of rows before removing duplicates: 12558
Count of rows after removing duplicates: 5125


In [5]:
df = df.drop(columns=['URL', 'Title'])


In [6]:

data = df
for column in data.columns:
    if data[column].dtype == 'object':
        data[column] = data[column].fillna(data[column].mode()[0])
    else:
        data[column] = data[column].fillna(data[column].median())

In [7]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
# Encode categorical features
label_encoders = {}
for column in data.select_dtypes(include='object').columns:
    le = LabelEncoder()
    data[column] = le.fit_transform(data[column])
    label_encoders[column] = le


In [8]:
# Split the data into features and target
X = data.drop(columns=['Price'])
y = data['Price']

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [10]:
# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
# Initialize and train XGBoost Regressor
model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [12]:
# Make predictions
y_pred = model.predict(X_test)

In [13]:
# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")


Mean Squared Error: 6296773913879.064
R-squared: 0.5435838651718541


In [14]:
# Feature importance
feature_importance = model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)

print("\nFeature Importances:")
print(feature_importance_df)


Feature Importances:
                   Feature  Importance
16                    Make    0.194269
12         Number of seats    0.139970
10               Body Type    0.136921
5        Registration city    0.122275
9                    Model    0.086941
14                   Color    0.048165
0                  Version    0.047396
13        Number of Owners    0.044317
4                 Location    0.044262
7                 Assembly    0.039232
11           Car documents    0.029982
3                 Interior    0.026652
15        Air Conditioning    0.016567
1               Power (hp)    0.010020
2   Consumption (l/100 km)    0.006584
6          Number of doors    0.003581
8                   Source    0.002867


In [ ]:
# Assuming these are the feature names used during training
train_columns = [
    'Make', 'Number of seats', 'Body Type', 'Registration city', 'Model',
    'Color', 'Version', 'Number of Owners', 'Location', 'Assembly', 'Car documents',
    'Interior', 'Air Conditioning', 'Power (hp)', 'Consumption (l/100 km)', 
    'Number of doors', 'Source'
]

# Convert new input to dataframe with same columns
new_input_df = pd.DataFrame([new_input], columns=train_columns)

# Ensure that all categorical features are encoded using the same encoders as during training
for column in new_input_df.select_dtypes(include='object').columns:
    le = label_encoders.get(column)
    if le:
        try:
            new_input_df[column] = le.transform(new_input_df[column])
        except ValueError:
            # Handle unseen category by setting it to the first class
            new_input_df[column] = le.transform([le.classes_[0]])

# 2. Standardize numerical features using the same scaler
new_input_scaled = scaler.transform(new_input_df)

# 3. Make the prediction
predicted_price = model.predict(new_input_scaled)

# Print the predicted price
print(f"Predicted Price: {predicted_price[0]}")

# If you know the actual price, you can compare the prediction
actual_price = 12000  # Replace with the actual price for this input

# Calculate the absolute error to evaluate accuracy
absolute_error = abs(predicted_price[0] - actual_price)
print(f"Actual Price: {actual_price}")
print(f"Absolute Error: {absolute_error}")


ValueError: The feature names should match those that were passed during fit.
Feature names must be in the same order as they were in fit.
